# Qwen/Qwen3.5-9B — RTX 5060 Ti 16GB VQA 비교용

**TASK-004 / EXP-011 / v1.0 — 작성·정적 검사 완료, 실제 GPU 실행 전**

`Run All` 흐름: 환경 설치 → 고정 버전 모델 파일 다운로드 → 데이터 점검·공통 분할 → 원본 모델 검증 → NF4 + LoRA 1 epoch → 어댑터 저장·재로드 → 검증 → test 추론 → `submission.csv`.

처음에는 **설정 셀의 DATA_DIR**을 수정하세요. 동일 GPU에서는 네 노트북을 동시에 실행하지 마세요. 운영체제의 NVIDIA 드라이버, Python 3.10~3.13, Jupyter, 인터넷 연결(준비 단계), 충분한 디스크 공간이 필요합니다. 권장 Python 3.11/3.12. CUDA toolkit/FlashAttention 별도 빌드는 요구하지 않습니다. GPU 메모리 적합성은 실제 데이터에서 확인해야 합니다.

전용 Python 가상환경에서 실행하므로 Notebook 커널을 재시작할 필요가 없습니다. 최초 설치·전체 가중치 다운로드는 오래 걸릴 수 있습니다. 4비트는 **로드 시 양자화**이며 원본 가중치 파일 다운로드 용량을 4비트로 줄이지 않습니다.

**대회 규칙:** 모든 답변은 PC GPU에서 직접 생성합니다. 외부 추론 API·온라인 데모·사용자 데이터 전송·Kaggle 자동 업로드 코드가 없습니다. 준비 단계는 패키지와 고정 revision 모델 파일만 다운로드하며, 학습·추론 프로세스는 HF 오프라인 설정과 Python socket 차단을 적용합니다. dev 응답은 사용하지 않습니다. test는 마지막 추론·제출 형식 확인에만 사용하고 학습/모델 선택에는 사용하지 않습니다.

**비교 주의:** 기본값은 기존 200→180/20 분할입니다. 실제 실행 시 같은 이미지의 train/valid 중복이 발견되면 중단합니다. 데이터 담당자의 `id,split` 파일을 네 노트북에 공통 적용하는 것이 우선입니다. 20개 검증으로 최종 우열을 확정하지 마세요. 모델별 공식 템플릿·이미지 processor는 다르며 이를 설정 파일에 기록합니다. 사용자 보고 0.703/0.83을 이 실행의 측정값으로 사용하지 않습니다.

공식 자료: [Qwen/Qwen3.5-9B](https://huggingface.co/Qwen/Qwen3.5-9B)



## 1. 모델 고정값과 공통 설정

같은 비교에서는 seed, 분할, 학습률, epoch, 프롬프트를 유지하세요. 첫 학습은 1 epoch이며 이전 프로젝트 어댑터를 덮어쓰거나 이어 학습하지 않습니다.


In [1]:
MODEL_ID = 'Qwen/Qwen3.5-9B'
MODEL_KIND = 'qwen'
MODEL_SLUG = 'Qwen3_5_9B'
EXPERIMENT_ID = 'EXP-011'
MODEL_REVISION = 'c202236235762e1c871ad0ccb60c8ee5ba337b9a'
MODEL_FILES = ['chat_template.jinja', 'config.json', 'merges.txt', 'model.safetensors-00001-of-00004.safetensors', 'model.safetensors-00002-of-00004.safetensors', 'model.safetensors-00003-of-00004.safetensors', 'model.safetensors-00004-of-00004.safetensors', 'model.safetensors.index.json', 'preprocessor_config.json', 'tokenizer.json', 'tokenizer_config.json', 'video_preprocessor_config.json', 'vocab.json']
TRANSFORMERS_VERSION = '5.8.0'
PEFT_VERSION = '0.18.1'
IMAGE_POLICY = 'Qwen 픽셀 예산 384², 종횡비 유지·processor 정렬'


In [2]:
from pathlib import Path
import sys, os, json, hashlib, subprocess, time, uuid

# 처음 실행 전에 이 경로만 실제 data/ 위치로 맞추세요.
# DATA_DIR 안: train.csv, test.csv, sample_submission.csv, train/, test/
DATA_DIR = Path("data").resolve()
PROJECT_DIR = Path.cwd().resolve()

# 데이터 담당자가 확정한 공통 분할이 있으면 같은 CSV를 네 노트북에 지정합니다.
# 형식: id,split  / split 값은 train 또는 valid. None이면 baseline의 200→180/20.
SPLIT_CSV = None
SEED = 42
TRAIN_SAMPLE_N = 200
VALID_N = 20

# 이미 준비된 파일만 사용하려면 둘 다 False. 학습/추론은 항상 오프라인입니다.
INSTALL_PACKAGES = True
DOWNLOAD_MODEL_FILES = True

if not (3, 10) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError("Python 3.10~3.13을 사용하세요. 권장: Python 3.11/3.12.")
for name in ["train.csv", "test.csv", "sample_submission.csv"]:
    if not (DATA_DIR / name).is_file():
        raise FileNotFoundError(f"DATA_DIR 설정을 확인하세요: {DATA_DIR / name}")
if SPLIT_CSV is not None:
    SPLIT_CSV = str(Path(SPLIT_CSV).resolve())
    if not Path(SPLIT_CSV).is_file():
        raise FileNotFoundError(SPLIT_CSV)

MODEL_DIR = PROJECT_DIR / "downloads" / "models" / MODEL_SLUG / MODEL_REVISION
ENV_DIR = PROJECT_DIR / "downloads" / "envs" / (MODEL_SLUG + "_v1")
ENV_PYTHON = ENV_DIR / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
RUN_DIR = PROJECT_DIR / "output" / "TASK-004" / EXPERIMENT_ID / (
    time.strftime("%Y%m%d_%H%M%S", time.gmtime()) + "_" + uuid.uuid4().hex[:8])
RUN_DIR.mkdir(parents=True, exist_ok=False)
CFG = {
    "task_id":"TASK-004", "experiment_id":EXPERIMENT_ID,
    "model_id":MODEL_ID, "kind":MODEL_KIND, "revision":MODEL_REVISION,
    "model_dir":str(MODEL_DIR), "data_dir":str(DATA_DIR), "run_dir":str(RUN_DIR),
    "split_csv":SPLIT_CSV, "seed":SEED, "train_sample_n":TRAIN_SAMPLE_N, "valid_n":VALID_N,
    "image_policy":IMAGE_POLICY, "epochs":1, "batch_size":1, "gradient_accumulation":4,
    "learning_rate":1e-4, "max_grad_norm":1.0, "max_new_tokens":2,
    "max_input_tokens":4096, "quantization":"NF4/double-quant/BF16-compute",
    "loss":"full_text; padding/media/role-special tokens masked; EOS supervised",
    "evaluation":"greedy2; parse failures explicitly logged + a-d constrained fallback",
    "code_version":"TASK-004-v1.0", "api_inference":False, "automatic_submission":False,
}
(RUN_DIR / "run_config.json").write_text(json.dumps(CFG, ensure_ascii=False, indent=2), encoding="utf-8")
print("모델:", MODEL_ID)
print("결과 폴더:", RUN_DIR)



모델: Qwen/Qwen3.5-9B
결과 폴더: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-004\EXP-011\20260921_052420_1293e54e


## 2. 전용 실행 환경 설치

이 단계만 pip 네트워크 다운로드를 허용합니다. 학습 데이터를 읽거나 전송하지 않습니다. 설치 로그·실제 패키지 버전은 결과 폴더에 저장합니다. NVIDIA 드라이버는 별도 준비가 필요합니다.


In [3]:
import venv

def run_command(args, log_name):
    # No shell; no credentials in command arguments or logs.
    with open(RUN_DIR / log_name, "a", encoding="utf-8") as log:
        proc = subprocess.Popen([str(x) for x in args], stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
        if proc.wait() != 0:
            raise RuntimeError(f"명령 실행 실패. {RUN_DIR / log_name} 확인 필요.")

DEPENDENCIES = [
    "transformers==" + TRANSFORMERS_VERSION, "peft==" + PEFT_VERSION,
    "accelerate==1.12.0", "bitsandbytes==0.49.2", "pandas==2.3.3", "numpy==2.2.6",
    "pillow==12.1.0", "safetensors>=0.6.2,<1", "sentencepiece>=0.2,<1", "protobuf>=5,<7",
    "timm==1.0.24", "einops==0.8.1", "psutil>=6,<8", "tqdm>=4.67,<5",
]
INSTALL_SPEC = {"torch":"2.11.0", "torchvision":"0.26.0", "cuda_index":"cu128",
                "dependencies":DEPENDENCIES, "python":sys.version_info[:2]}
spec_text = json.dumps(INSTALL_SPEC, sort_keys=True)
spec_hash = hashlib.sha256(spec_text.encode()).hexdigest()
receipt = ENV_DIR / "install_spec.sha256"
if INSTALL_PACKAGES:
    if not ENV_PYTHON.exists():
        venv.EnvBuilder(with_pip=True).create(ENV_DIR)
    if not receipt.exists() or receipt.read_text().strip() != spec_hash:
        run_command([ENV_PYTHON, "-m", "pip", "install", "--upgrade", "pip"], "install.log")
        run_command([ENV_PYTHON, "-m", "pip", "install", "torch==2.11.0", "torchvision==0.26.0",
                     "--index-url", "https://download.pytorch.org/whl/cu128"], "install.log")
        run_command([ENV_PYTHON, "-m", "pip", "install", *DEPENDENCIES], "install.log")
        run_command([ENV_PYTHON, "-m", "pip", "check"], "install.log")
        receipt.write_text(spec_hash, encoding="utf-8")
    else:
        print("동일 설정의 전용 환경을 재사용합니다.")
elif not ENV_PYTHON.exists() or not receipt.exists() or receipt.read_text().strip() != spec_hash:
    raise RuntimeError("동일한 전용 실행 환경을 먼저 준비하세요. INSTALL_PACKAGES=True 필요.")

freeze = subprocess.check_output([str(ENV_PYTHON), "-m", "pip", "freeze"], text=True, encoding="utf-8")
(RUN_DIR / "requirements.lock.txt").write_text(freeze, encoding="utf-8")
(RUN_DIR / "install_spec.json").write_text(spec_text, encoding="utf-8")
print("실행 환경:", ENV_PYTHON)



   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   -- ------------------------------------- 0.1/1.8 MB 1.7 MB/s eta 0:00:02
   ---------------------------------- ----- 1.6/1.8 MB 14.3 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 12.9 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.0
    Uninstalling pip-24.0:
      Successfully uninstalled pip-24.0
Looking in indexes: https://download.pytorch.org/whl/cu128
  Using cached torch-2.11.0%2Bcu128-cp311-cp311-win_amd64.whl.metadata (29 kB)
  Using cached torchvision-0.26.0%2Bcu128-cp311-cp311-win_amd64.whl.metadata (5.6 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Usin

## 3. 모델 파일 다운로드

공식 저장소의 고정 commit 파일 주소에 HTTPS GET만 사용합니다. 외부 서비스에 추론 요청을 보내지 않습니다. 성공한 파일의 SHA256을 보존하며 재실행 시 확인 후 재사용합니다. 다운로드가 중단되면 성공한 파일은 유지되고 미완료 파일만 다시 받습니다.


In [4]:
# 다운로드 전용: 모델 저장소의 고정 revision 파일을 HTTPS GET으로 받습니다.
# 외부 추론 API, HfApi, InferenceClient, snapshot_download, 사용자 데이터 업로드 없음.
import urllib.request
import urllib.parse
import urllib.error
import getpass
import shutil

class SafeRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        if urllib.parse.urlparse(newurl).scheme != "https":
            raise RuntimeError("HTTPS 이외의 다운로드 리디렉션을 거부합니다.")
        redirected = super().redirect_request(req, fp, code, msg, headers, newurl)
        if redirected is not None and urllib.parse.urlparse(newurl).hostname != "huggingface.co":
            redirected.remove_header("Authorization")
        return redirected

def hash_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda:f.read(4*1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

MODEL_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = MODEL_DIR / "download_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
records = manifest.get("files", {}) if manifest.get("revision") == MODEL_REVISION else {}
pending = []
for name in MODEL_FILES:
    p = MODEL_DIR / name
    record = records.get(name, {})
    valid = (p.is_file() and p.stat().st_size == record.get("size")
             and hash_file(p) == record.get("sha256"))
    if not valid:
        pending.append(name)
if pending and not DOWNLOAD_MODEL_FILES:
    raise FileNotFoundError("모델 파일 준비가 필요합니다: " + ", ".join(pending))

token = None
if pending and MODEL_KIND == "gemma":
    token = os.environ.get("HF_TOKEN") or getpass.getpass(
        "Gemma 사용 동의를 마친 Hugging Face read token (화면/파일에 저장하지 않음): ").strip()
    if not token:
        raise RuntimeError("Gemma 다운로드에는 모델 사용 동의와 read token이 필요합니다.")
opener = urllib.request.build_opener(SafeRedirect())
try:
    for name in pending:
        if Path(name).name != name:
            raise ValueError("예상하지 않은 모델 파일 경로")
        url = "https://huggingface.co/" + MODEL_ID + "/resolve/" + MODEL_REVISION + "/" + name
        headers = {"User-Agent":"local-vqa-asset-download/1.0"}
        if token:
            headers["Authorization"] = "Bearer " + token
        target = MODEL_DIR / name
        partial = target.with_name(target.name + ".part")
        print("다운로드:", name, flush=True)
        for attempt in range(3):
            try:
                request = urllib.request.Request(url, headers=headers)
                digest = hashlib.sha256()
                size = 0
                with opener.open(request, timeout=120) as response, open(partial, "wb") as f:
                    if "text/html" in response.headers.get("Content-Type", ""):
                        raise RuntimeError("파일 대신 HTML 응답을 받았습니다.")
                    expected = response.headers.get("Content-Length")
                    last_report = time.monotonic()
                    for chunk in iter(lambda:response.read(4*1024*1024), b""):
                        f.write(chunk)
                        digest.update(chunk)
                        size += len(chunk)
                        if time.monotonic()-last_report > 15:
                            print(f"  {name}: {size/2**30:.2f} GiB", flush=True)
                            last_report = time.monotonic()
                if size == 0 or (expected and size != int(expected)):
                    raise RuntimeError("다운로드 파일 길이 불일치")
                partial.replace(target)
                records[name] = {"size":size,"sha256":digest.hexdigest()}
                manifest_path.write_text(json.dumps({"model_id":MODEL_ID, "revision":MODEL_REVISION,
                                                    "files":records}, indent=2), encoding="utf-8")
                break
            except urllib.error.HTTPError as exc:
                if exc.code in (401,403):
                    raise RuntimeError("모델 다운로드 권한이 없습니다. 모델 사용 승인·read token을 확인하세요.") from None
                if attempt == 2:
                    raise RuntimeError(f"{name}: HTTP {exc.code} 다운로드 실패") from None
            except (OSError, RuntimeError) as exc:
                if attempt == 2:
                    raise RuntimeError(f"{name}: 다운로드 실패 ({type(exc).__name__}). 네트워크/디스크를 확인하세요.") from None
finally:
    token = None
    if "headers" in globals():
        headers.pop("Authorization", None)
    if "request" in globals():
        request.remove_header("Authorization")
shutil.copy2(manifest_path, RUN_DIR / "model_assets.json")
print("모델 파일 준비 완료. 이후 학습·추론은 네트워크 차단 상태로 실행합니다.")



다운로드: chat_template.jinja
다운로드: config.json
다운로드: merges.txt
다운로드: model.safetensors-00001-of-00004.safetensors
  model.safetensors-00001-of-00004.safetensors: 0.90 GiB
  model.safetensors-00001-of-00004.safetensors: 1.71 GiB
  model.safetensors-00001-of-00004.safetensors: 2.36 GiB
  model.safetensors-00001-of-00004.safetensors: 3.03 GiB
  model.safetensors-00001-of-00004.safetensors: 3.76 GiB
  model.safetensors-00001-of-00004.safetensors: 4.53 GiB
다운로드: model.safetensors-00002-of-00004.safetensors
  model.safetensors-00002-of-00004.safetensors: 0.87 GiB
  model.safetensors-00002-of-00004.safetensors: 1.84 GiB
  model.safetensors-00002-of-00004.safetensors: 2.87 GiB
  model.safetensors-00002-of-00004.safetensors: 3.75 GiB
  model.safetensors-00002-of-00004.safetensors: 4.76 GiB
다운로드: model.safetensors-00003-of-00004.safetensors
  model.safetensors-00003-of-00004.safetensors: 0.49 GiB
  model.safetensors-00003-of-00004.safetensors: 1.10 GiB
  model.safetensors-00003-of-00004.safetensor

## 4. 로컬 실행 코드 정의

아래 셀은 독립 실행 프로세스에 전달할 Python 소스를 구성합니다. 모두 실행한 뒤 5단계에서 실제 GPU 작업이 시작됩니다. 별도의 .py 파일을 함께 내려받을 필요가 없습니다.


In [5]:
WORKER_PARTS = []


### 데이터·중복·제출 검사 및 네트워크 차단


In [6]:
WORKER_PARTS.append(r'''
import argparse
import csv
import gc
import hashlib
import json
import math
import os
import random
import re
import sys
import time
import traceback
from pathlib import Path


def write_json(path, obj):
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding="utf-8")


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1024 * 1024), b""):
            h.update(b)
    return h.hexdigest()


def block_network():
    # HF flags alone do not cover all third-party code. Also reject Python socket access.
    os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
                      HF_DATASETS_OFFLINE="1", HF_HUB_DISABLE_TELEMETRY="1",
                      WANDB_DISABLED="true", TOKENIZERS_PARALLELISM="false",
                      CUBLAS_WORKSPACE_CONFIG=":4096:8")
    os.environ.pop("HF_TOKEN", None)
    def guard(event, args):
        if event in {"socket.connect", "socket.getaddrinfo", "socket.sendto"}:
            raise RuntimeError("학습·추론 프로세스의 네트워크 호출은 금지되어 있습니다.")
    sys.addaudithook(guard)


def read_table(path, columns):
    import pandas as pd
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    missing = set(columns) - set(df.columns)
    if missing:
        raise ValueError(f"{path}: 필수 컬럼 누락 {sorted(missing)}")
    if df.empty or df.id.eq("").any() or df.id.duplicated().any():
        raise ValueError(f"{path}: 빈 데이터, 빈 ID 또는 중복 ID")
    for col in columns:
        if df[col].str.strip().eq("").any():
            raise ValueError(f"{path}: {col}에 빈 값이 있습니다.")
    return df


def resolve_image(root, value):
    # CSV paths are local relative paths. Never fetch image URLs.
    value = str(value).replace("\\", "/")
    if "://" in value or Path(value).is_absolute() or ":" in value:
        raise ValueError(f"이미지는 DATA_DIR 내부 상대 경로여야 합니다: {value}")
    root = Path(root).resolve()
    p = (root / value).resolve()
    if not p.is_relative_to(root) or not p.is_file():
        raise FileNotFoundError(f"이미지 경로 확인 필요: {p}")
    return p


def build_mc_prompt(row):
    return (f"{row['question']}\n"
            f"(a) {row['a']}\n(b) {row['b']}\n(c) {row['c']}\n(d) {row['d']}\n\n"
            "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.")


SYSTEM_INSTRUCT = ("You are a helpful visual question answering assistant. "
                   "Answer using exactly one letter among a, b, c, or d. No explanation.")


def parse_answer(raw):
    # Do not extract arbitrary letters from prose or default to a.
    s = str(raw).strip()
    s = re.sub(r"^(?:answer|정답)\s*[:：]\s*", "", s, flags=re.I)
    m = re.fullmatch(r"(?:\(([a-d])\)|([a-d]))[.。]?", s, flags=re.I)
    return (m.group(1) or m.group(2)).lower() if m else None


def prepare_data(cfg, out):
    import pandas as pd
    from PIL import Image, ImageOps
    root = Path(cfg["data_dir"])
    cols = ["id", "path", "question", "a", "b", "c", "d"]
    train = read_table(root / "train.csv", cols + ["answer"])
    test = read_table(root / "test.csv", cols)
    sample = read_table(root / "sample_submission.csv", ["id"])
    if set(sample.columns) != {"id", "answer"}:
        raise ValueError("sample_submission.csv 컬럼은 id, answer여야 합니다.")
    if not train.answer.isin(list("abcd")).all():
        raise ValueError("train answer는 공백 없는 소문자 a~d여야 합니다.")
    if set(sample.id) != set(test.id) or len(sample) != len(test):
        raise ValueError("sample_submission/test ID 집합·행 수가 다릅니다.")
    if set(train.id) & set(test.id):
        raise ValueError("train/test ID가 겹칩니다. 데이터 담당자 확인 필요.")
    for df in (train, test):
        for row in df.to_dict("records"):
            resolve_image(root, row["path"])
    if cfg["split_csv"]:
        split = read_table(cfg["split_csv"], ["id", "split"])
        if not split.split.isin(["train", "valid"]).all() or set(split.split) != {"train", "valid"}:
            raise ValueError("공통 split CSV는 id, split(train/valid) 형식이어야 합니다.")
        if not set(split.id) <= set(train.id):
            raise ValueError("분할 파일에 train.csv에 없는 ID가 있습니다.")
    else:
        n, nvalid = cfg["train_sample_n"], cfg["valid_n"]
        if not 0 < nvalid < n <= len(train):
            raise ValueError(f"{n}→{n-nvalid}/{nvalid} 분할 불가. 데이터 수={len(train)}. 설정을 명시적으로 수정하세요.")
        selected = train.sample(n=n, random_state=cfg["seed"]).reset_index(drop=True)
        split = pd.DataFrame({"id": selected.id,
                              "split": ["train"] * (n-nvalid) + ["valid"] * nvalid})
    split.to_csv(out / "split_manifest.csv", index=False)
    indexed = train.set_index("id", drop=False)
    tr = indexed.loc[split.loc[split.split.eq("train"), "id"]].reset_index(drop=True)
    va = indexed.loc[split.loc[split.split.eq("valid"), "id"]].reset_index(drop=True)
    # Exact decoded-pixel hashes detect re-encoded copies. Near-duplicates still need independent review.
    hashes = {}
    image_rows = []
    for label, df in [("train", tr), ("valid", va)]:
        for row in df.to_dict("records"):
            p = resolve_image(root, row["path"])
            with Image.open(p) as im:
                im = ImageOps.exif_transpose(im).convert("RGB")
                h = hashlib.sha256(str(im.size).encode() + im.tobytes()).hexdigest()
                image_rows.append({"id":row["id"], "split":label, "path":row["path"],
                                   "pixel_sha256":h, "width":im.width, "height":im.height})
                hashes.setdefault(h, set()).add(label)
    pd.DataFrame(image_rows).to_csv(out / "image_audit.csv", index=False)
    if any(len(groups) > 1 for groups in hashes.values()):
        raise ValueError("학습/검증에 같은 픽셀의 이미지가 있습니다. image_audit.csv를 확인하고 공통 분할을 수정하세요.")
    info = {"train_csv_sha256":sha256(root / "train.csv"),
            "test_csv_sha256":sha256(root / "test.csv"),
            "sample_csv_sha256":sha256(root / "sample_submission.csv"),
            "split_sha256":sha256(out / "split_manifest.csv"),
            "train_n":len(tr), "valid_n":len(va), "test_n":len(test),
            "near_duplicate_review":"not_performed", "dev_used":False}
    write_json(out / "data_manifest.json", info)
    print("분할:", info, flush=True)
    return tr, va, test, sample, info


def make_submission(test, sample, predictions, path):
    import pandas as pd
    pred = pd.DataFrame(predictions)
    if pred.empty or pred.id.duplicated().any() or set(pred.id) != set(test.id):
        raise ValueError("예측 ID 중복·누락·추가가 있습니다.")
    if not pred.answer.isin(list("abcd")).all():
        raise ValueError("유효하지 않은 예측값이 있습니다.")
    submission = sample[["id"]].merge(pred[["id", "answer"]], on="id", how="left", validate="one_to_one")
    submission = submission.loc[:, list(sample.columns)]
    assert len(submission) == len(test) and submission.id.tolist() == sample.id.tolist()
    submission.to_csv(path, index=False, encoding="utf-8")
    reread = pd.read_csv(path, dtype=str, keep_default_na=False)
    assert reread.equals(submission)
    return submission
''')


### 모델별 입력 처리·NF4·LoRA·어댑터 재로드


In [7]:
WORKER_PARTS.append(r'''
def move_tensors(x, device, dtype):
    import torch
    if torch.is_tensor(x):
        return x.to(device=device, dtype=dtype if x.is_floating_point() else x.dtype)
    if isinstance(x, dict):
        return {k:move_tensors(v, device, dtype) for k,v in x.items()}
    if isinstance(x, (tuple, list)):
        return [move_tensors(v, device, dtype) for v in x]
    return x


class ModelAdapter:
    def __init__(self, cfg, out):
        import torch
        from transformers import AutoProcessor, BitsAndBytesConfig
        self.cfg, self.out = cfg, out
        self.kind = cfg["kind"]
        self.dtype = torch.bfloat16
        self.device = torch.device("cuda:0")
        source = cfg["model_dir"]
        if not torch.cuda.is_bf16_supported():
            raise RuntimeError("이 비교 설정은 BF16을 지원하는 CUDA GPU가 필요합니다.")
        trust = self.kind == "minicpm"
        self.processor = AutoProcessor.from_pretrained(source, trust_remote_code=trust, local_files_only=True)
        self.tokenizer = self.processor.tokenizer
        # All options must be representable by one token for the explicit fallback.
        self.choice_ids = [self.tokenizer.encode(c, add_special_tokens=False) for c in "abcd"]
        if not all(len(x) == 1 for x in self.choice_ids) or len({x[0] for x in self.choice_ids}) != 4:
            raise ValueError("선지 토큰이 단일·고유 토큰이 아닙니다. 추론 코드 검토 필요.")
        self.choice_ids = [x[0] for x in self.choice_ids]
        qconfig = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=self.dtype,
                    llm_int8_skip_modules=["visual", "vision_tower", "vision_model", "vpm",
                                           "resampler", "multi_modal_projector", "mlp1", "lm_head"])
        kwargs = dict(local_files_only=True, device_map={"":"cuda:0"},
                      torch_dtype=self.dtype, quantization_config=qconfig, attn_implementation="sdpa")
        if self.kind == "minicpm":
            from transformers import AutoModel
            self.model = AutoModel.from_pretrained(source, trust_remote_code=True, **kwargs)
        else:
            from transformers import AutoModelForImageTextToText
            self.model = AutoModelForImageTextToText.from_pretrained(source, **kwargs)
        self.base = self.model
        self.model.eval()
        self.configure_processor()
        self.model.config.use_cache = False
        self.input_logged = False
        self.processor.save_pretrained(out / "processor")
        write_json(out / "model_config.json", self.base.config.to_dict())
        write_json(out / "processor_settings.json", {
            "kind":self.kind, "image_policy":cfg["image_policy"],
            "thinking":False, "choice_ids":self.choice_ids,
            "image_processor":self.processor.image_processor.to_dict()})

    def configure_processor(self):
        ip = self.processor.image_processor
        if self.kind == "qwen":
            # Qwen3.5 uses Qwen3VLProcessor / Qwen2VLImageProcessor.
            ip.size = {"shortest_edge":384*384, "longest_edge":384*384}
            if hasattr(ip, "min_pixels"):
                ip.min_pixels = 384*384
            if hasattr(ip, "max_pixels"):
                ip.max_pixels = 384*384
        elif self.kind == "internvl":
            ip.crop_to_patches = False
            ip.min_patches = 1
            ip.max_patches = 1
        elif self.kind == "gemma":
            ip.do_pan_and_scan = False
        # MiniCPM max_slice_nums=1 is passed per call without altering config invariants.

    def encode(self, row, training=False):
        import torch
        from PIL import Image, ImageOps
        p = resolve_image(self.cfg["data_dir"], row["path"])
        with Image.open(p) as f:
            image = ImageOps.exif_transpose(f).convert("RGB")
        prompt = build_mc_prompt(row)
        if self.kind == "minicpm":
            messages = [{"role":"system", "content":SYSTEM_INSTRUCT},
                        {"role":"user", "content":"(<image>./</image>)\n" + prompt}]
            if training:
                messages.append({"role":"assistant", "content":row["answer"]})
            text = self.tokenizer.apply_chat_template(messages, tokenize=False,
                       add_generation_prompt=not training, enable_thinking=False)
            inputs = dict(self.processor([text], [[image]], max_slice_nums=1,
                          return_tensors="pt", max_length=None))
            inputs.pop("image_sizes", None)
            inputs["input_ids"] = inputs["input_ids"].long()
        else:
            # Gemma doesn't accept an independent system role; place the same instruction in user text.
            messages = [] if self.kind == "gemma" else [
                {"role":"system", "content":[{"type":"text", "text":SYSTEM_INSTRUCT}]}]
            user_text = SYSTEM_INSTRUCT + "\n\n" + prompt if self.kind == "gemma" else prompt
            messages.append({"role":"user", "content":[{"type":"image"}, {"type":"text", "text":user_text}]})
            if training:
                messages.append({"role":"assistant", "content":[{"type":"text", "text":row["answer"]}]})
            text = self.processor.apply_chat_template(messages, tokenize=False,
                        add_generation_prompt=not training, enable_thinking=False)
            inputs = dict(self.processor(text=[text], images=[image], return_tensors="pt", padding=False,
                                         add_special_tokens=False))
        if inputs["input_ids"].shape[1] > self.cfg["max_input_tokens"]:
            raise ValueError(f"ID {row['id']}: 입력 길이 초과. 조용히 자르지 않습니다.")
        if training:
            labels = inputs["input_ids"].clone()
            # Full textual sequence supervision; ignore padding/media/role special tokens.
            labels[inputs["attention_mask"] == 0] = -100
            for sid in self.tokenizer.all_special_ids:
                if sid != self.tokenizer.eos_token_id:
                    labels[inputs["input_ids"] == sid] = -100
            for bound in inputs.get("image_bound", []):
                for start, stop in bound.tolist():
                    labels[:, start:stop] = -100
            inputs["labels"] = labels
        if not self.input_logged:
            def shape(x):
                if torch.is_tensor(x): return {"shape":list(x.shape),"dtype":str(x.dtype)}
                if isinstance(x,list): return [shape(y) for y in x]
                return str(type(x).__name__)
            write_json(self.out / "first_input_shapes.json", {k:shape(v) for k,v in inputs.items()})
            self.input_logged = True
        return move_tensors(inputs, self.device, self.dtype)

    def add_lora(self):
        import torch
        from peft import LoraConfig, get_peft_model
        # Equivalent k-bit preparation without temporarily doubling large frozen embeddings
        # in FP32. That transient allocation can exceed a 16 GiB card before being cast back.
        for name, p in self.model.named_parameters():
            p.requires_grad_(False)
            if "norm" in name.lower() and p.is_floating_point() and p.ndim == 1:
                p.data = p.data.to(torch.float32)
        self.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False})
        self.model.enable_input_require_grads()
        suffixes = {"q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"}
        excluded = {"visual","vision_tower","vision_model","vpm","resampler","multi_modal_projector","mlp1"}
        targets = [name for name, m in self.model.named_modules()
                   if name.split(".")[-1] in suffixes and not (set(name.split(".")) & excluded)
                   and hasattr(m, "weight")]
        if not targets:
            raise RuntimeError("언어 모듈 LoRA 대상이 없습니다.")
        config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules=targets,
                            bias="none", task_type=None if self.kind == "minicpm" else "CAUSAL_LM")
        self.model = get_peft_model(self.model, config)
        self.base = self.model.get_base_model()
        if self.kind == "minicpm":
            self.base.llm.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False})
        self.model.print_trainable_parameters()
        write_json(self.out / "lora_targets.json", targets)

    def loss(self, inputs):
        import torch
        if self.kind != "minicpm":
            return self.model(**inputs, use_cache=False).loss
        # MiniCPM has a custom data-dict forward; keep its vision/resampler frozen.
        with torch.no_grad():
            embeddings, _ = self.base.get_vllm_embedding(inputs)
        embeddings = embeddings.detach().requires_grad_(True)
        return self.base.llm(inputs_embeds=embeddings, attention_mask=inputs["attention_mask"],
                             labels=inputs["labels"], use_cache=False).loss

    def generate(self, inputs, constrained=False):
        import torch
        kwargs = dict(max_new_tokens=1 if constrained else self.cfg["max_new_tokens"],
                      do_sample=False, num_beams=1, repetition_penalty=1.0, use_cache=True)
        if constrained:
            kwargs["prefix_allowed_tokens_fn"] = lambda batch_id, input_ids: self.choice_ids
        with torch.inference_mode(), torch.autocast("cuda", dtype=self.dtype):
            if self.kind == "minicpm":
                embeddings, _ = self.base.get_vllm_embedding(inputs)
                ids = self.base.llm.generate(inputs_embeds=embeddings,
                        attention_mask=inputs["attention_mask"], pad_token_id=0,
                        eos_token_id=[self.tokenizer.convert_tokens_to_ids(t) for t in self.base.terminators],
                        **kwargs)
                # inputs_embeds-only generation returns generated token IDs, not the input text.
                generated = ids[0]
            else:
                ids = self.model.generate(**inputs, **kwargs)
                generated = ids[0, inputs["input_ids"].shape[1]:]
        return self.tokenizer.decode(generated, skip_special_tokens=True).strip()

    def reload_adapter(self, directory):
        from peft import PeftModel
        # Unload adapter only, preserving the already loaded 4-bit base to avoid a second full model.
        base = self.model.unload()
        self.model = PeftModel.from_pretrained(base, str(directory), local_files_only=True, is_trainable=False)
        self.base = self.model.get_base_model()
        self.model.eval()
''')


### 정답 없는 생성 평가·학습 루프


In [8]:
WORKER_PARTS.append(r'''
def evaluate(adapter, df, label, out, has_answers=True):
    import pandas as pd
    import torch
    adapter.model.eval()
    started = time.perf_counter()
    torch.cuda.reset_peak_memory_stats()
    result = []
    # Stream predictions to disk to preserve completed rows on interruption.
    fields = ["id", "answer", "raw_output", "parse_failed", "fallback_output", "gold", "correct", "strict_correct"]
    with open(out / f"{label}_predictions.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for i, row in enumerate(df.to_dict("records")):
            # Explicitly remove gold before preparing every evaluation prompt.
            clean = {k:v for k,v in row.items() if k != "answer"}
            inputs = adapter.encode(clean, training=False)
            raw = adapter.generate(inputs)
            parsed = parse_answer(raw)
            answer, fallback = parsed, ""
            if answer is None:
                fallback = adapter.generate(inputs, constrained=True)
                answer = parse_answer(fallback)
                if answer is None:
                    raise RuntimeError(f"ID {row['id']}: 제한 생성도 a~d를 반환하지 않았습니다.")
            gold = row["answer"] if has_answers else ""
            rec = dict(id=row["id"], answer=answer, raw_output=raw, parse_failed=parsed is None,
                       fallback_output=fallback, gold=gold,
                       correct=(answer == gold) if has_answers else "",
                       strict_correct=(parsed == gold) if has_answers else "")
            result.append(rec)
            writer.writerow(rec)
            f.flush()
            del inputs
            if i % 25 == 0 or i+1 == len(df):
                print(f"{label}: {i+1}/{len(df)}", flush=True)
    seconds = time.perf_counter() - started
    metrics = {"n":len(result), "seconds":seconds, "seconds_per_sample":seconds/len(result),
               "parse_failure_rate":sum(r["parse_failed"] for r in result)/len(result),
               "peak_allocated_gib":torch.cuda.max_memory_allocated()/2**30,
               "peak_reserved_gib":torch.cuda.max_memory_reserved()/2**30}
    if has_answers:
        metrics.update(accuracy=sum(r["correct"] for r in result)/len(result),
                       strict_accuracy=sum(r["strict_correct"] for r in result)/len(result))
    write_json(out / f"{label}_metrics.json", metrics)
    print(label, metrics, flush=True)
    return result, metrics


def train_one_epoch(adapter, df, cfg, out):
    import torch
    from transformers import get_linear_schedule_with_warmup
    params = [p for p in adapter.model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=cfg["learning_rate"], weight_decay=0.01)
    updates = math.ceil(len(df)/cfg["gradient_accumulation"])
    scheduler = get_linear_schedule_with_warmup(optimizer, int(updates*0.03), updates)
    records = df.to_dict("records")
    random.Random(cfg["seed"]).shuffle(records)
    adapter.model.train()
    optimizer.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    rows = []
    grad_checked = False
    # Correct normalization for the final incomplete accumulation group.
    for group_start in range(0, len(records), cfg["gradient_accumulation"]):
        group = records[group_start:group_start+cfg["gradient_accumulation"]]
        total_loss = 0.0
        for row in group:
            inputs = adapter.encode(row, training=True)
            with torch.autocast("cuda", dtype=adapter.dtype):
                loss = adapter.loss(inputs)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"ID {row['id']}: loss가 NaN/Inf입니다.")
            raw_loss = float(loss.detach())
            (loss / len(group)).backward()
            total_loss += raw_loss
            del inputs, loss
        grads = [p.grad for p in params if p.grad is not None]
        if not grads or not all(bool(torch.isfinite(g).all()) for g in grads):
            raise FloatingPointError("LoRA gradient가 없거나 NaN/Inf입니다.")
        if not grad_checked:
            if not any(bool(g.abs().max() > 0) for g in grads):
                raise RuntimeError("LoRA gradient가 모두 0입니다.")
            write_json(out / "backward_smoke.json", {"finite_loss":True,"nonzero_lora_gradient":True})
            grad_checked = True
        grad_norm = float(torch.nn.utils.clip_grad_norm_(params, cfg["max_grad_norm"]))
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        rows.append({"update":len(rows)+1, "mean_loss":total_loss/len(group),
                     "grad_norm":grad_norm, "lr":scheduler.get_last_lr()[0]})
        print(f"train {len(rows)}/{updates}, loss={rows[-1]['mean_loss']:.4f}", flush=True)
    import pandas as pd
    pd.DataFrame(rows).to_csv(out / "train_log.csv", index=False)
    metrics = {"epochs":1,"updates":len(rows),"seconds":time.perf_counter()-start,
               "peak_allocated_gib":torch.cuda.max_memory_allocated()/2**30,
               "peak_reserved_gib":torch.cuda.max_memory_reserved()/2**30}
    write_json(out / "train_metrics.json", metrics)
    del optimizer, scheduler, params, grads
    gc.collect()
    torch.cuda.empty_cache()
    return metrics
''')


### 공통 실행 순서·상태 기록


In [9]:
WORKER_PARTS.append(r'''
def main(cfg):
    block_network()
    import torch
    import numpy as np
    from importlib.metadata import version
    out = Path(cfg["run_dir"])
    write_json(out / "status.json", {"state":"started","gpu_executed":False})
    tr, va, test, sample, data_info = prepare_data(cfg, out)
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU를 찾을 수 없습니다. 드라이버와 PyTorch CUDA 설치를 확인하세요.")
    torch.manual_seed(cfg["seed"])
    torch.cuda.manual_seed_all(cfg["seed"])
    random.seed(cfg["seed"])
    np.random.seed(cfg["seed"])
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=True)
    # A short actual CUDA computation catches missing Blackwell kernel support.
    x = torch.ones((32,32), device="cuda", dtype=torch.bfloat16)
    assert float((x @ x)[0,0]) == 32.0
    del x
    env = {"python":sys.version,"torch":torch.__version__,"cuda":torch.version.cuda,
           "gpu":torch.cuda.get_device_name(0),"capability":torch.cuda.get_device_capability(0),
           "vram_gib":torch.cuda.get_device_properties(0).total_memory/2**30,
           "packages":{p:version(p) for p in ["transformers","peft","bitsandbytes","accelerate"]},
           "network":"blocked_in_training_and_inference_process",
           "determinism":"seed_fixed; nondeterministic-kernel warnings retained in log"}
    write_json(out / "environment.json", env)
    print(env, flush=True)
    adapter = ModelAdapter(cfg, out)
    _, baseline = evaluate(adapter, va, "valid_base", out)
    adapter.add_lora()
    training = train_one_epoch(adapter, tr, cfg, out)
    checkpoint = out / "adapter_epoch1"
    adapter.model.save_pretrained(checkpoint)
    adapter.processor.save_pretrained(checkpoint)
    gc.collect()
    torch.cuda.empty_cache()
    # Reload saved adapter bytes before producing validation and submission predictions.
    adapter.reload_adapter(checkpoint)
    _, tuned = evaluate(adapter, va, "valid_lora", out)
    # Keep selection explicit: this experiment submits the epoch-1 LoRA checkpoint.
    # Base and LoRA validation results are both reported; no Public-driven selection.
    predictions, inference = evaluate(adapter, test, "test", out, has_answers=False)
    submission = make_submission(test, sample, predictions, out / "submission.csv")
    summary = {"task_id":"TASK-004", "experiment_id":cfg["experiment_id"],
               "model":cfg["model_id"], "revision":cfg["revision"], "split":data_info,
               "baseline":baseline,"lora":tuned,"training":training,"test":inference,
               "submitted_checkpoint":"adapter_epoch1", "submission_rows":len(submission),
               "submission_sha256":sha256(out / "submission.csv"), "kaggle_uploaded":False,
               "public_score":None,
               "selection_note":"epoch-1 LoRA inference; model adoption awaits common-split review",
               "base_outperformed_lora":baseline["accuracy"] > tuned["accuracy"]}
    write_json(out / "summary.json", summary)
    write_json(out / "status.json", {"state":"completed","gpu_executed":True,"kaggle_uploaded":False})
    print("완료:", out / "submission.csv", flush=True)
    print("Accuracy: base=", baseline["accuracy"], "LoRA=", tuned["accuracy"], flush=True)


if __name__ == "__main__":
    config_path = Path(sys.argv[1])
    cfg = json.loads(config_path.read_text(encoding="utf-8"))
    try:
        main(cfg)
    except Exception as exc:
        write_json(Path(cfg["run_dir"]) / "status.json", {
            "state":"failed", "error_type":type(exc).__name__,"error":str(exc),
            "note":"실패한 실행의 부분 출력은 최종 제출물로 사용하지 마세요."})
        traceback.print_exc()
        sys.exit(1)
''')


## 5. 학습·검증·추론 실행

`valid_base_predictions.csv`와 `valid_lora_predictions.csv`에 원문·실패·정오를 남깁니다. 파싱 실패는 임의로 a로 바꾸지 않고, a~d 토큰으로 제한한 추가 생성으로 처리합니다. 자유 생성만의 strict Accuracy와 fallback 포함 Accuracy를 함께 기록합니다. 제출 CSV는 **저장 후 재로드한 epoch-1 LoRA** 결과입니다. 이 노트북은 모델 채택이나 최종 제출을 자동 결정하지 않습니다.


In [10]:
worker = RUN_DIR / "run_local_vqa.py"
source = "\n\n".join(WORKER_PARTS)
compile(source, str(worker), "exec")
worker.write_text(source, encoding="utf-8")
CFG["worker_sha256"] = hash_file(worker)
(RUN_DIR / "run_config.json").write_text(json.dumps(CFG, ensure_ascii=False, indent=2), encoding="utf-8")
print("Run All: 데이터 점검 → base 검증 → LoRA 1 epoch → 저장/재로드 → 검증 → test CSV")
run_command([ENV_PYTHON, "-u", worker, RUN_DIR / "run_config.json"], "run.log")



Run All: 데이터 점검 → base 검증 → LoRA 1 epoch → 저장/재로드 → 검증 → test CSV
분할: {'train_csv_sha256': '83b31c210e42f9c47aa992960dfb5e5519cc2fdea089c82158ed2d73b1fa985b', 'test_csv_sha256': '95063b713a853153d80d7c10df95a5be2bac93bb87cb6a7fe6172c479dc1bb97', 'sample_csv_sha256': '92567292544cba39e2ed81780ba7627758a518f7925fb237d350901a2d9cdace', 'split_sha256': '12a0551f8af9dc9ac77a01f1c1e878c420aacde5dd67524b87d3d5d8c44437dc', 'train_n': 180, 'valid_n': 20, 'test_n': 6714, 'near_duplicate_review': 'not_performed', 'dev_used': False}
{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'capability': (12, 0), 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked_in_training_and_inference_process', 'determinism': 'seed_fixed; nondeterministic-kernel warnings retained in log'}
[tra

## 6. 결과 확인

성공한 실행만 summary.json과 submission.csv가 생성됩니다. 실패 시 status.json과 run.log를 확인하세요. GPU 메모리 부족 시 설정을 자동으로 바꾸지 않습니다. 해상도/분할/양자화 변경은 새 실험으로 기록하고 동일 조건의 비교를 다시 수행하세요.


In [ ]:
summary = json.loads((RUN_DIR / "summary.json").read_text(encoding="utf-8"))
print("Base Accuracy:", summary["baseline"]["accuracy"])
print("LoRA Accuracy:", summary["lora"]["accuracy"])
print("LoRA 파싱 실패율:", summary["lora"]["parse_failure_rate"])
print("결과 CSV:", RUN_DIR / "submission.csv")
print("Kaggle 업로드: 수행하지 않음")
if summary["base_outperformed_lora"]:
    print("주의: 이번 검증에서는 base가 LoRA보다 높았습니다. 제출 CSV는 epoch-1 LoRA 결과입니다.")
if summary["split"]["valid_n"] == 20:
    print("검증 20문항은 1문항당 5%p입니다. 이 결과만으로 최종 모델을 선정하지 마세요.")
from IPython.display import display, FileLink
display(FileLink(str(RUN_DIR / "submission.csv")))
display(FileLink(str(RUN_DIR / "summary.json")))



Base Accuracy: 0.9
LoRA Accuracy: 0.9
LoRA 파싱 실패율: 0.0
결과 CSV: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-004\EXP-011\20260921_052420_1293e54e\submission.csv
Kaggle 업로드: 수행하지 않음
검증 20문항은 1문항당 5%p입니다. 이 결과만으로 최종 모델을 선정하지 마세요.


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-004\EXP-011\20260921_052420_1293e54e\submission.csv

C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-004\EXP-011\20260921_052420_1293e54e\summary.json

: 

## 결과 전달 체크

- `run_config.json`, `requirements.lock.txt`, `model_assets.json`: 코드·패키지·모델 revision/파일 해시.
- `split_manifest.csv`, `data_manifest.json`, `image_audit.csv`: 분할 ID와 데이터 버전, 정확히 동일한 이미지 검사. 유사 이미지·같은 촬영 장면 검토는 별도입니다.
- `valid_*_metrics.json`, `valid_*_predictions.csv`, `train_metrics.json`, `test_metrics.json`: Accuracy·실패율·시간·VRAM.
- `adapter_epoch1/`, `processor/`: 저장된 LoRA와 processor. 베이스 모델은 `downloads/models/`에 별도 보존합니다.
- `run_local_vqa.py`, `environment.json`, `run.log`, `status.json`, `summary.json`, `submission.csv`.

같은 분할 해시·설정인지 확인한 뒤 네 모델을 비교하세요. Public 점수만으로 모델을 채택하지 말고, 분할·지표·loss 변경과 최고 모델 교체·최종 제출 시 02 검토를 요청합니다. Kaggle 업로드는 사용자가 별도 진행하며 팀당 하루 20회 한도를 함께 관리합니다. 이 노트북이 다른 팀원의 제출 횟수를 확인하지는 못합니다.

원본 baseline 대비 공통 수정: 생성 부분만 디코딩, 생성 Accuracy 추가, 특수/이미지/pad 토큰 loss 제외, 올바른 loss 표시·마지막 gradient 누적 처리, BF16 통일·gradient clipping, 네트워크 차단, 저장 어댑터 재로드와 제출 형식 검사. 따라서 이 결과를 기존 보고 점수와 단순히 '모델만 바꾼 효과'로 해석하지 마세요.

